# General Fine-tuning Model based on Qwen3-4B (Baseline C)

In [ ]:
# Run this on Remote Jupyter Book.

import os

os.getcwd()
# os.chdir("/root/OtakuLab")

## Define Training Set Format

In [18]:
from typing_extensions import TypedDict
from typing import Optional
from pathlib import Path
import json
import re

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents | dict[str, list[str]|dict]
    output: str

class DatasetStorage:
    def __init__(self) -> None:
        self.items: dict[str, DatasetItem] = {}

    def __len__(self) -> int:
        return len(self.items)

    def new_item(self, character: str, neutral_sentence: str, output: str):
        item = DatasetItem(character=character,
                           neutral_sentence=neutral_sentence,
                           instruction_components={},
                           output=output)
        self.items[output] = item

    def save_character_signature_rules(self, character: str, signature_rules: list[str]) -> None:
        saved = False
        for item in self.items.values():
            if item["character"] == character:
                item["instruction_components"]["signature_rules"] = signature_rules
                saved = True
        if not saved:
            raise ValueError(f"角色 {character} 未找到")   

    def save_component(
        self,
        output: str,
        structural_metrics: Optional[dict[str, float]] = None,
        semantic_tags: Optional[list[str]] = None,
        signature_rules: Optional[list[str]] = None,
        # 兼容 v1 参数名
        syntactic_vector: Optional[dict[str, float]] = None,
        pragmatic_styles: Optional[list[str]] = None,
    ) -> None:
        if output not in self.items:
            raise ValueError(f"风格句: {output} 似乎未加载")

        if structural_metrics is None and syntactic_vector is not None:
            structural_metrics = syntactic_vector
        if semantic_tags is None and pragmatic_styles is not None:
            semantic_tags = pragmatic_styles

        comp = self.items[output]["instruction_components"]
        if structural_metrics is not None:
            comp["structural_metrics"] = structural_metrics
        if semantic_tags is not None:
            comp["semantic_tags"] = semantic_tags
        if signature_rules is not None:
            comp["signature_rules"] = signature_rules
    
    @staticmethod
    def _verify_validity(item: DatasetItem):
        if not all((item['character'],
                    item['neutral_sentence'],
                    item['output'],
                    item['instruction_components'])):
            return False
        
        instruction_components = item['instruction_components']
        
        return all((instruction_components.get('semantic_tags', None),
                    instruction_components.get('signature_rules', None)))

    def output(self, output_path: Path, oversampling:bool = True):
        items: list[DatasetItem] = list(self.items.values())
        valid_items = []
        vaild_items_count = 0
        
        for item in items:
            if not self._verify_validity(item):
                continue

            instr = item["instruction_components"]
            semantic_tags = ", ".join(instr.get("semantic_tags", [])) or "None"
            signature_rules = " ".join(instr.get("signature_rules", [])) or "None"
            user_prompt = (
                f"Target Character: {item['character']}\n"
                f"Semantic Traits: {semantic_tags}\n"
                f"Formatting Signature: {signature_rules}\n"
                f"Neutral Content: {item['neutral_sentence']}\n"
            )

            # 构建 ShareGPT 格式
            conversation = {
                "conversations": [
                    {
                        "from": "human",
                        "value": user_prompt
                    },
                    {
                        "from": "gpt",
                        "value": item['output']
                    }
                ],
                "system":  (
                    "You are an expert in stylistic rewriting. "
                    "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
                )
            }

            valid_items.append(conversation)
            vaild_items_count += 1

            if oversampling:
                difficult_labels_5x = {"tsundere", "sharp_tongued", "proud"}
                difficult_labels_3x = {"tsukkomi", "chuunibyou", "yandere", "airhead"}

                semantic_tags = set(item["instruction_components"]["semantic_tags"])

                if semantic_tags & difficult_labels_5x:
                    for _ in range(5):
                        valid_items.append(conversation)
                elif semantic_tags & difficult_labels_3x:
                    for _ in range(3):
                        valid_items.append(conversation)


        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(valid_items, f, ensure_ascii=False, indent=2)

        print(f"训练集已导出至 {output_path}, 总训练集数量: {len(valid_items)}(非过采样数量: {vaild_items_count}), 跳过数量: {len(items) - vaild_items_count}")

dataset_storage = DatasetStorage()

## Load Neutral Sentences

In [19]:
neutral_sentences_jsonl_file = Path("./data/neutral_sentences_with_CoT.jsonl")

EN_NAME_TO_ZH = {"Muice": "沐雪",
                 "Ayaka": "神里绫华",
                 "Zhongli": "钟离",
                 "Hutao": "胡桃",
                 "Haruhi": "凉宫春日",
                 "Liyunlong": "李云龙",
                 "Sheldon": "谢尔顿",
                 "Wukong": "孙悟空",
                 }

class NSFileItem(TypedDict):
    character: str
    original: str
    neutral: str
    CoT: str

def load_jsonl_file(jsonl_file: Path):
    with open(jsonl_file, "r", encoding="utf-8") as f:
        jsonl_file_lines = f.readlines()

    for line in jsonl_file_lines:
        if line := line.rstrip():
            item: NSFileItem = json.loads(line)
            character = EN_NAME_TO_ZH.get(item["character"], item["character"])
            neutral = item["neutral"]
            dataset_storage.new_item(character, neutral, item["original"])

load_jsonl_file(neutral_sentences_jsonl_file)
print(f"成功加载了 {len(dataset_storage)} 条训练集条目")

成功加载了 10835 条训练集条目


## Process Lexical Layer Vectors

In [21]:
def get_lexical_keywords_from_file(file: Path, top_n: int = 25) -> list[str]:
    with open(file, "r", encoding="utf-8") as f:
        data: dict[str, float] = json.loads(f.read())
    return list(data.keys())[:top_n]

def load_punctuation_signature(file: Path) -> dict[str, list[str]]:
    rows = json.loads(file.read_text(encoding="utf-8"))
    return {row["role"]: row.get("S_signature", []) for row in rows}

def build_signature_rules(lexical_keywords: list[str], punctuation_symbols: list[str]) -> list[str]:
    rules: list[str] = []
    if lexical_keywords:
        rules.append("Lexical cues: " + ", ".join(lexical_keywords[:25]))
    if punctuation_symbols:
        rules.append("Punctuation tendencies: " + ", ".join(punctuation_symbols[:10]))
    return rules

LEXICAL_DIR = Path("./outputs/style/signature/lexical")
PUNCTUATION_SIGNATURE_FILE = Path("./outputs/style/signature/punctuation/symbol_signature.json")

CHARACTER_TO_FILE_STEM = {
    "沐雪": "muice",
    "神里绫华": "ayaka",
    "钟离": "zhongli",
    "胡桃": "hutao",
    "凉宫春日": "haruhi",
    "李云龙": "liyunlong",
    "谢尔顿": "sheldon",
    "孙悟空": "wukong"
}

punctuation_signature_map = load_punctuation_signature(PUNCTUATION_SIGNATURE_FILE)

for character, file_stem in CHARACTER_TO_FILE_STEM.items():
    lexical_file = LEXICAL_DIR / f"{file_stem}_pmi_filtered.json"
    lexical_keywords = get_lexical_keywords_from_file(lexical_file, top_n=25)
    punctuation_symbols = punctuation_signature_map.get(character, [])

    signature_rules = build_signature_rules(lexical_keywords, punctuation_symbols)
    dataset_storage.save_character_signature_rules(character, signature_rules)

print("成功注入 Signature 层（Lexical + Punctuation）规则")

成功注入 Signature 层（Lexical + Punctuation）规则


## Process Dense Structural Metrics (v_dense)

### Load Dense Metrics from Normalized Style Vectors

In [22]:
DENSE_STYLE_FILE = Path("./outputs/style/dense/normalized/style_vector_phase2_minmax.json")

def _to_float(value: object, default: float = 0.0) -> float:
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return default

def load_dense_metrics_by_role(file_path: Path) -> dict[str, dict[str, float]]:
    rows = json.loads(file_path.read_text(encoding="utf-8"))
    role_to_metrics: dict[str, dict[str, float]] = {}

    for row in rows:
        role = row.get("role")
        if not role:
            continue

        role_to_metrics[role] = {
            "mdd": _to_float(row.get("mdd", row.get("MDD", 0.0))),
            "verb_noun_ratio": _to_float(row.get("verb_noun_ratio", 0.0)),
            "question_ratio": _to_float(row.get("question_ratio", 0.0)),
            "imperative_ratio": _to_float(row.get("imperative_ratio", 0.0)),
            "fragment_ratio": _to_float(row.get("fragment_ratio", 0.0)),
            "marker_density": _to_float(row.get("marker_density", 0.0)),
        }

    return role_to_metrics

dense_role_metrics = load_dense_metrics_by_role(DENSE_STYLE_FILE)
print(f"成功加载 {len(dense_role_metrics)} 个角色的稠密结构向量")

TARGET_CHARACTERS = ["沐雪", "神里绫华", "钟离", "胡桃", "凉宫春日", "李云龙", "谢尔顿", "孙悟空"]

for character in TARGET_CHARACTERS:
    if character not in dense_role_metrics:
        raise ValueError(f"稠密向量缺失角色: {character}")

    metrics = dense_role_metrics[character]
    updated = 0
    for item in dataset_storage.items.values():
        if item["character"] == character:
            dataset_storage.save_component(item["output"], structural_metrics=metrics)
            updated += 1

    print(f"角色 {character} 已注入 6 维 dense 结构向量，样本数: {updated}")

print("成功注入 Dense 层结构向量")

成功加载 9 个角色的稠密结构向量
角色 沐雪 已注入 6 维 dense 结构向量，样本数: 2667
角色 神里绫华 已注入 6 维 dense 结构向量，样本数: 1410
角色 钟离 已注入 6 维 dense 结构向量，样本数: 506
角色 胡桃 已注入 6 维 dense 结构向量，样本数: 834
角色 凉宫春日 已注入 6 维 dense 结构向量，样本数: 1093
角色 李云龙 已注入 6 维 dense 结构向量，样本数: 1775
角色 谢尔顿 已注入 6 维 dense 结构向量，样本数: 1850
角色 孙悟空 已注入 6 维 dense 结构向量，样本数: 700
成功注入 Dense 层结构向量


## Process Semantic Style Layer Vectors

In [ ]:
PRAGMATIC_DIR = Path("./outputs/style/semantic/prag_vectors")

CHARACTER_TO_PRAG_FILE = {
    "沐雪": "muice.jsonl",
    "神里绫华": "ayaka.jsonl",
    "钟离": "zhongli.jsonl",
    "胡桃": "hutao.jsonl",
    "凉宫春日": "haruhi.jsonl",
    "李云龙": "liyunlong.jsonl",
    "谢尔顿": "sheldon.jsonl",
    "孙悟空": "wukong.jsonl"
}

class RawPragmaticItem(TypedDict):
    prompt: str
    response: str
    pragmatic_styles: list[dict[str, float]]

class SemanticItem(TypedDict):
    response: str
    semantic_tags: list[str]

def read_semantic_jsonl_file(jsonl_file: Path, threshold: float = 0.4) -> list[SemanticItem]:
    items: list[SemanticItem] = []
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            raw_item: RawPragmaticItem = json.loads(line)
            score_map: dict[str, float] = {}
            for vec in raw_item.get("pragmatic_styles", []):
                score_map.update(vec)

            semantic_tags = [
                key for key, value in score_map.items()
                if float(value) > threshold
            ]
            items.append({"response": raw_item["response"], "semantic_tags": semantic_tags})

    return items

updated = 0
skipped = 0

for _, file_name in CHARACTER_TO_PRAG_FILE.items():
    semantic_items = read_semantic_jsonl_file(PRAGMATIC_DIR / file_name, threshold=0.4)
    for item in semantic_items:
        try:
            dataset_storage.save_component(item["response"], semantic_tags=item["semantic_tags"])
            updated += 1
        except ValueError:
            skipped += 1

print(f"更新了 {updated} 条语义标签，跳过 {skipped} 条未匹配 response")

更新了 11031 条语义标签，跳过 714 条未匹配 response


## Export Training Set File

In [ ]:
OUTPUT_PATH = Path("./data/vanilla_llm_train.json")

dataset_storage.output(OUTPUT_PATH)

训练集已导出至 data\vanilla_llm_train.json, 总训练集数量: 10179(非过采样数量: 8753), 跳过数量: 2082


: 